<a href="https://colab.research.google.com/github/TiagoRodriguesskt/Fisica-Computacional/blob/main/arcabou_o_matematico_de_mode_omega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalações de bibliotecas
import sympy as sp
import numpy as np

In [ ]:
class ModelOmega:
    """
    Implementação técnica do arcabouço matemático do Model-Omega.
    Foco em simulação cosmológica e análise de estabilidade.
    """
    def __init__(self, params):
        # 23. ESPAÇO DE PARÂMETROS (Vetor P)
        self.eta = params.get('eta', 0.1)
        self.beta = params.get('beta', 0.05)
        self.xi = params.get('xi', 0.1)
        self.V0 = params.get('V0', 1.0)  # eV^4
        self.Omega_t = params.get('Omega_t', 0.5)
        self.Delta = params.get('Delta', 0.01)
        self.m0 = params.get('m0', 1e-33)
        self.Omega_min = params.get('Omega_min', 1.0)
        self.V_lambda = params.get('V_lambda', 1e-47) # GeV^4

        # Constantes Fundamentais
        self.M_pl = 1.0  # Unidades de Planck reduzidas
        self.Lambda_cutoff = 1.0 # Escala de corte EFT
        self.G = 1.0

    # 15. POTENCIAL MÍNIMO FUNCIONAL CORRIGIDO
    def V(self, Omega):
        term_ede = self.V0 * (1 + np.tanh((Omega - self.Omega_t) / self.Delta))
        term_mass = 0.5 * self.m0**2 * (Omega - self.Omega_min)**2 * \
                    (0.5 * (1 + np.tanh((Omega - self.Omega_t) / self.Delta)))
        return term_ede + term_mass + self.V_lambda

    # 4. MASSA EFETIVA DO CAMPO
    def m_eff_sq(self, Omega, R):
        # Derivada numérica simples do potencial (V'')
        dOmega = 1e-5
        v_pp = (self.V(Omega + dOmega) - 2*self.V(Omega) + self.V(Omega - dOmega)) / (dOmega**2)
        return v_pp - self.xi * R

    # 13. VELOCIDADE DO SOM DAS PERTURBAÇÕES
    def speed_of_sound_sq(self, X):
        num = 1 + (8 * self.eta / self.Lambda_cutoff**2) * X
        den = 1 + (24 * self.eta / self.Lambda_cutoff**2) * X
        return num / den

    # 8. FATOR DE TRIAGEM (SCREENING)
    def screening_S(self, a, k, Omega, R):
        m_sq = self.m_eff_sq(Omega, R)
        return 1 / (1 + (a**2 * m_sq) / k**2)

    # 10. EQUAÇÃO DE POISSON MODIFICADA (G_eff)
    def G_eff(self, a, k, Omega, R):
        S = self.screening_S(a, k, Omega, R)
        return self.G * (1 + 2 * self.beta**2 * S)

    # 21. ANISOTROPIA ESCALAR (SLIP)
    def gravitational_slip(self, Omega, dOmega):
        num = self.xi * Omega * dOmega
        den = self.M_pl**2 - self.xi * Omega**2
        return num / den

    # 7. DENSIDADE DE ENERGIA ESCALAR (rho_Omega)
    def rho_Omega(self, Omega, Omega_dot, H):
        X = 0.5 * Omega_dot**2
        term1 = X
        term2 = (9 * self.eta / self.Lambda_cutoff**2) * X**2
        term3 = self.V(Omega)
        term4 = -6 * self.xi * H * Omega * Omega_dot
        term5 = -3 * self.xi * H**2 * Omega**2
        return term1 + term2 + term3 + term4 + term5

    # 11. EQUAÇÃO DE CRESCIMENTO (ODE Step)
    def growth_evolution(self, delta_m, delta_m_dot, a, k, Omega, R, rho_m, H):
        geff = self.G_eff(a, k, Omega, R)
        # dd_delta + 2H*d_delta - 4*pi*Geff*rho*delta = 0
        return -2 * H * delta_m_dot + 4 * np.pi * geff * rho_m * delta_m

    # 14. REGIMES DO SOM (Condicionais)
    def sound_regime_check(self, z, X):
        if z > 1e6:
            return 1/3  # BBN/Radiação
        elif z < 10:
            return 1.0  # Era Tardia
        else:
            return self.speed_of_sound_sq(X)

    # 17. ESTABILIDADE EFT (Check)
    def is_eft_stable(self, Omega, R):
        return np.sqrt(np.abs(self.m_eff_sq(Omega, R))) < self.Lambda_cutoff

    # 2. VÍNCULOS GW (Boolean Check)
    def check_gw_constrains(self, G4_X, G5):
        return G4_X == 0 and G5 == 0 # c_T^2 = 1 garantido

    # 26. LIMITE GR / 27. LIMITE LCDM (Reduções)
    def get_limits(self, mode='GR'):
        if mode == 'GR':
            return {'beta': 0, 'xi': 0}
        if mode == 'LCDM':
            return {'Omega_dot': 0, 'V': self.V_lambda}

    # 30. CONDIÇÃO GLOBAL DE VALIDADE
    def global_validity(self, Omega, R, X, z):
        conditions = [
            self.is_eft_stable(Omega, R),
            0 <= self.speed_of_sound_sq(X) <= 1,
            self.xi > 1/12 if z > 1e8 else True # BBN check
        ]
        return all(conditions)

# --- Exemplo de uso numérico ---
params_test = {
    'eta': 0.2, 'beta': 0.04, 'xi': 0.15,
    'V0': 2.0, 'Omega_t': 0.6, 'Delta': 0.05
}

model = ModelOmega(params_test)

# Testando a densidade de energia no atrator (Ponto 7)
rho = model.rho_Omega(Omega=0.6, Omega_dot=0.01, H=67.0)
print(f"Densidade de Energia Escalar (Ponto 7): {rho}")

# Testando a supressão de S8 (Ponto 20 - Lógica de redução)
delta_S8 = 0.1 * model.beta # Proporcional ao acoplamento
print(f"Supressão Delta S8 (Ponto 20): {delta_S8}")

Densidade de Energia Escalar (Ponto 7): -725.5797499954998
Supressão Delta S8 (Ponto 20): 0.004


# 1 --> Ação Integral (Horndeski Customizada)

In [ ]:
# 1. Ativa a renderização visual matemática (Essencial para Colab/Jupyter)
sp.init_printing()

# 2. Definição de Símbolos e Constantes (Usando nomes em LaTeX para símbolos gregos)
M_Pl, xi, eta, Lambda, g_det, R, X, S_m = sp.symbols(r'M_{Pl} \xi \eta \Lambda g R X S_m')
d4x = sp.symbols(r'd^4x')

# 3. Definição do campo Omega e seus índices
# Definimos x como símbolo para ser o argumento da função
x = sp.symbols('x')
Omega = sp.Function(r'\Omega')(x)
mu, nu = sp.symbols(r'\mu \nu')
g_inv = sp.Symbol(r'g^{\mu \nu}')

# Derivadas parciais
d_mu_Omega = sp.Derivative(Omega, mu)
d_nu_Omega = sp.Derivative(Omega, nu)

# 4. Construção dos termos da Lagrangiana
termo_1 = (M_Pl**2 / 2) * R
termo_2 = (sp.Rational(1, 2)) * xi * (Omega**2) * R
termo_3 = -(sp.Rational(1, 2)) * g_inv * d_mu_Omega * d_nu_Omega
termo_4 = -(3 * eta / Lambda**2) * (X**2)
termo_5 = -sp.Function('V')(Omega)

lagrangiana = termo_1 + termo_2 + termo_3 + termo_4 + termo_5

# 5. Montagem da Ação Integral S
# Nota: Usamos sqrt(-g_det) para representar o determinante da métrica
S = sp.Integral(sp.sqrt(-g_det) * lagrangiana, d4x) + S_m

# 6. Exibição final
# No Colab, não use print() ou pprint() se quiser a fonte matemática bonita.
# Apenas coloque a variável na última linha.
S

     ⌠                                                                         ↪
     ⎮        ⎛                                                         d      ↪
     ⎮        ⎜      2                 2         2        g__{\mu \nu}⋅────(\O ↪
     ⎮   ____ ⎜M_{Pl} ⋅R   R⋅\xi⋅\Omega (x)   3⋅X ⋅\eta                d\mu    ↪
Sₘ + ⎮ ╲╱ -g ⋅⎜───────── + ──────────────── - ───────── - ──────────────────── ↪
     ⎮        ⎜    2              2                  2                         ↪
     ⎮        ⎝                               \Lambda                          ↪
     ⌡                                                                         ↪

↪                                                  
↪           d                            ⎞         
↪ mega(x))⋅────(\Omega(x))               ⎟         
↪          d\nu                          ⎟         
↪ ──────────────────────── - V(\Omega(x))⎟ d(d__4x)
↪  2                                     ⎟         
↪                                      

# 2 --> Vínculos de Estabilidade

In [ ]:
import sympy as sp

# 1. Definição de Símbolos
# G4_X representa a derivada de G4 em relação a X (G_{4,X})
G4_X = sp.Symbol('G_{4,X}')
G5 = sp.Symbol('G_5')
cT_2 = sp.Symbol('c_T^2')

# 2. Definição dos Vínculos (Equações)
vinculo1 = sp.Eq(G4_X, 0)
vinculo2 = sp.Eq(G5, 0)
vinculo3 = sp.Eq(cT_2, 1)

# Exibição
print("Vínculos de Estabilidade (GW170817):")
display(vinculo1, vinculo2, vinculo3)


Vínculos de Estabilidade (GW170817):


Eq(G_{4,X}, 0)

Eq(G_5, 0)

Eq(c_T^2, 1)

# 3 --> Equação de Klein-Gordon (Fundo)

In [ ]:
# 1. Ativa a renderização matemática visual no Colab
sp.init_printing()

# 2. Definição de Símbolos e Funções (usando r'' para renderizar LaTeX)
t = sp.symbols('t')
M_Pl, xi, beta, rho_m = sp.symbols(r'M_{Pl} \xi \beta \rho_m')

# Funções que dependem do tempo
Omega = sp.Function(r'\Omega')(t)
H = sp.Function('H')(t)
V = sp.Function('V')(Omega)

# 3. Derivadas em relação ao tempo
Omega_dot = Omega.diff(t)
Omega_ddot = Omega.diff(t, 2)
H_dot = H.diff(t)

# Derivada do Potencial em relação a Omega (V')
V_prime = V.diff(Omega)

# 4. Construção da Equação
# Lado Esquerdo: \ddot{\Omega} + 3H\dot{\Omega} + V'
lado_esquerdo = Omega_ddot + 3 * H * Omega_dot + V_prime
# Lado Direito: 6\xi(2H^2 + \dot{H})\Omega + \beta(\rho_m/M_{Pl})
lado_direito = 6 * xi * (2 * H**2 + H_dot) * Omega + beta * (rho_m / M_Pl)

# 5. Definindo o objeto da Equação
kg_equacao = sp.Eq(lado_esquerdo, lado_direito)

# 6. Exibição: No Colab, apenas chame a variável na última linha
kg_equacao

                                                   2                           ↪
       d                   d                      d                      ⎛   2 ↪
3⋅H(t)⋅──(\Omega(t)) + ──────────(V(\Omega(t))) + ───(\Omega(t)) = 6⋅\xi⋅⎜2⋅H  ↪
       dt              d\Omega(t)                   2                    ⎝     ↪
                                                  dt                           ↪

↪                                        
↪       d       ⎞             \beta⋅\rhoₘ
↪ (t) + ──(H(t))⎟⋅\Omega(t) + ───────────
↪       dt      ⎠               M_{Pl}   
↪                                        

# 4 --> Massa Efetiva do Campo

In [ ]:
sp.init_printing() # Isso ativa a renderização LaTeX visual

Omega = sp.symbols('Omega')
xi, R = sp.symbols('xi R')
V = sp.Function('V')(Omega)

m_eff_sq = sp.symbols('m_{eff}^2')
# Ao apenas digitar a variável no final da célula, ela aparece formatada
sp.Eq(m_eff_sq, V.diff(Omega, 2) - xi * R)

                     2       
                    d        
m_{eff}__2 = -R⋅ξ + ───(V(Ω))
                      2      
                    dΩ       

# 5 --> Dinâmica do Atrator (Slow-Roll Escalar)

In [ ]:
sp.init_printing() # Ativa o LaTeX no Colab/VS Code

# 1. Definição do Símbolo xi (usando r para string raw do LaTeX)
xi = sp.symbols(r'\xi')

# 2. Definição da Desigualdade (Vínculo do Atrator)
# O sp.Rational(1, 12) garante que o SymPy trate como fração exata
condicao_atrator = xi > sp.Rational(1, 12)

# Exibição
condicao_atrator

\xi > 1/12

# 7 --> Densidade de Energia Escalar ($\rho_\Omega$):

In [ ]:
sp.init_printing()

# 1. Definição de Símbolos e Funções
t = sp.symbols('t')
xi, eta, Lambda, H = sp.symbols(r'\xi \eta \Lambda H')
rho_Omega = sp.symbols(r'\rho_\Omega')

Omega = sp.Function(r'\Omega')(t)
V = sp.Function('V')(Omega)

# 2. Definição das Derivadas Temporais
Omega_dot = Omega.diff(t)

# 3. Definição de X (termo cinético no fundo)
X = (sp.Rational(1, 2)) * Omega_dot**2

# 4. Construção da Equação de Densidade de Energia
# Note o uso de sp.Rational para as frações 1/2 e os expoentes
termo_1 = (sp.Rational(1, 2)) * Omega_dot**2
termo_2 = (9 * eta / Lambda**2) * (X**2)
termo_3 = V
termo_4 = -6 * xi * H * Omega * Omega_dot
termo_5 = -3 * xi * (H**2) * (Omega**2)

equacao_rho = sp.Eq(rho_Omega, termo_1 + termo_2 + termo_3 + termo_4 + termo_5)

# Exibição
equacao_rho

                                                                               ↪
                                                                               ↪
                                                                               ↪
                   2           2                        d                      ↪
\rho_\Omega = - 3⋅H ⋅\xi⋅\Omega (t) - 6⋅H⋅\xi⋅\Omega(t)⋅──(\Omega(t)) + V(\Ome ↪
                                                        dt                     ↪
                                                                               ↪

↪                         2                         4
↪          ⎛d            ⎞           ⎛d            ⎞ 
↪          ⎜──(\Omega(t))⎟    9⋅\eta⋅⎜──(\Omega(t))⎟ 
↪          ⎝dt           ⎠           ⎝dt           ⎠ 
↪ ga(t)) + ──────────────── + ───────────────────────
↪                 2                          2       
↪                                   4⋅\Lambda        

# 8 --> Fator de Triagem (Screening)

In [ ]:
sp.init_printing()

# 1. Definição de Símbolos
a, k = sp.symbols('a k')
m_eff = sp.symbols('m_{eff}')
# Definindo S como uma função de (a, k)
S = sp.Function('S')(a, k)

# 2. Construção da Expressão
# Usamos sp.Rational para garantir a precisão simbólica se houver números
expressao_screening = 1 / (1 + (a**2 * m_eff**2) / k**2)

# 3. Montagem da Equação Final
equacao_S = sp.Eq(S, expressao_screening)

# Exibição
equacao_S

                 1       
S(a, k) = ───────────────
           2        2    
          a ⋅m_{eff}     
          ─────────── + 1
               2         
              k          

# 9 --> Potenciais no Gauge Newtoniano

In [ ]:
sp.init_printing()

# 1. Definição de Símbolos
a, k, beta, H = sp.symbols(r'a k \beta H')
Phi_GR, Psi_GR = sp.symbols(r'\Phi_{GR} \Psi_{GR}')
Phi, Psi = sp.symbols(r'\Phi \Psi')

# Função de screening S(a,k) e derivada de Omega
S_ak = sp.Function('S')(a, k)
Omega_dot = sp.symbols(r'\dot{\Omega}')

# 2. Construção das Equações
# Equação para Phi
eq_phi = sp.Eq(Phi, Phi_GR * (1 - 2 * beta**2 * S_ak))

# Equação para Psi
eq_psi = sp.Eq(Psi, Psi_GR * (1 + 2 * beta**2 * S_ak * (Omega_dot / H)))

# 3. Exibição
display(eq_phi, eq_psi)

                 ⎛         2            ⎞
\Phi = \Phi_{GR}⋅⎝- 2⋅\beta ⋅S(a, k) + 1⎠

                 ⎛           2                     ⎞
                 ⎜    2⋅\beta ⋅\dot{\Omega}⋅S(a, k)⎟
\Psi = \Psi_{GR}⋅⎜1 + ─────────────────────────────⎟
                 ⎝                  H              ⎠

# 10 --> Equação de Poisson Modificada

In [ ]:
sp.init_printing()

# 1. Definição de Símbolos
k, a, G, rho_m, delta_m = sp.symbols(r'k a G \rho_m \delta_m')
beta, m_eff, Phi = sp.symbols(r'\beta m_{eff} \Phi')

# 2. Construção do termo de correção (o fator de triagem está no denominador)
# S(a,k) = 1 / (1 + (a^2 * m_eff^2 / k^2))
correcao_quinta_forca = (2 * beta**2) / (1 + (a**2 * m_eff**2 / k**2))

# 3. Montagem da Equação de Poisson Modificada
# Lado esquerdo: k^2 * Phi
# Lado direito: 4 * pi * G * a^2 * rho_m * delta_m * [1 + correção]
lado_esquerdo = k**2 * Phi
lado_direito = 4 * sp.pi * G * a**2 * rho_m * delta_m * (1 + correcao_quinta_forca)

equacao_poisson = sp.Eq(lado_esquerdo, lado_direito)

# 4. Exibição
equacao_poisson

                                 ⎛          2        ⎞
      2                        2 ⎜   2⋅\beta         ⎟
\Phi⋅k  = 4⋅π⋅G⋅\deltaₘ⋅\rhoₘ⋅a ⋅⎜─────────────── + 1⎟
                                 ⎜ 2        2        ⎟
                                 ⎜a ⋅m_{eff}         ⎟
                                 ⎜─────────── + 1    ⎟
                                 ⎜     2             ⎟
                                 ⎝    k              ⎠